# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [52]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [53]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [54]:
#your code here
target_col = 'Transported'

X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [55]:
from sklearn.preprocessing import StandardScaler

# Identify numeric columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Initialize scaler
scaler = StandardScaler()

# Scale numeric features only
X[num_cols] = scaler.fit_transform(X[num_cols])


In [56]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns


In [57]:
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)


In [58]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_encoded[num_cols] = scaler.fit_transform(X_encoded[num_cols])


In [59]:
from sklearn.impute import SimpleImputer

# For numeric columns, fill missing with median
num_imputer = SimpleImputer(strategy='median')
X_encoded[num_cols] = num_imputer.fit_transform(X_encoded[num_cols])

# For categorical columns (one-hot encoded columns), fill missing with 0
# But since we already one-hot encoded, missing values in categorical features
# should be rare; still you can check or fillna with 0
X_encoded.fillna(0, inplace=True)


In [60]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=10)
X_selected = selector.fit_transform(X_encoded, y)

selected_features = X_encoded.columns[selector.get_support()]
print("Selected features:")
print(selected_features)


Selected features:
Index(['Age', 'RoomService', 'FoodCourt', 'Spa', 'VRDeck', 'HomePlanet_Europa',
       'CryoSleep_True', 'Cabin_G/981/S', 'Destination_TRAPPIST-1e',
       'VIP_True'],
      dtype='object')


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [ ]:
#your code here

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier  # or your best model

param_grid = {
    'n_estimators': [100, 200, 300],        # number of trees
    'learning_rate': [0.01, 0.1, 0.2],      # step size shrinkage
    'max_depth': [3, 5, 7],                  # max depth of each tree
    'subsample': [0.8, 1.0],                 # fraction of samples to fit each tree
    'min_samples_split': [2, 5, 10]          # minimum samples to split internal node
}


gb_model = GradientBoostingClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=gb_model,
    param_grid=param_grid,
    cv=5,               # 5-fold cross-validation
    n_jobs=-1,          # use all CPU cores
    scoring='accuracy'  # or another metric you prefer
)

grid_search.fit(X_train, y_train)

print("Best parameters found:", grid_search.best_params_)
best_model = grid_search.best_estimator_


2296.07s - Error patching args (debugger not attached to subprocess).
Traceback (most recent call last):
  File "/Users/abc/miniconda3/lib/python3.13/site-packages/debugpy/_vendored/pydevd/_pydev_bundle/pydev_monkey.py", line 541, in patch_args
    new_args.append(_get_python_c_args(host, port, code, unquoted_args, SetupHolder.setup))
                    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/abc/miniconda3/lib/python3.13/site-packages/debugpy/_vendored/pydevd/_pydev_bundle/pydev_monkey.py", line 193, in _get_python_c_args
    if "__future__" in code:
       ^^^^^^^^^^^^^^^^^^^^
TypeError: a bytes-like object is required, not 'str'
0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.04s -

- Evaluate your model

In [ ]:
#your code here
from sklearn.metrics import accuracy_score, classification_report

y_pred = best_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))



**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [ ]:
#your code here
param_grid = {
    'n_estimators': [100, 200, 300],        # number of trees
    'learning_rate': [0.01, 0.1, 0.2],      # step size shrinkage
    'max_depth': [3, 5, 7],                  # max depth of each tree
    'subsample': [0.8, 1.0],                 # fraction of samples to fit each tree
    'min_samples_split': [2, 5, 10]          # minimum samples to split internal node
}



- Run Grid Search

In [ ]:
#your code here
gb_model = GradientBoostingClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=gb_model,
    param_grid=param_grid,
    cv=5,               # 5-fold cross-validation
    n_jobs=-1,          # use all CPU cores
    scoring='accuracy'  # or another metric you prefer
)

grid_search.fit(X_train, y_train)

print("Best parameters found:", grid_search.best_params_)
best_model = grid_search.best_estimator_
   

- Evaluate your model

In [ ]:

   from sklearn.metrics import accuracy_score, classification_report

y_pred = best_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

